# Archived coursework code
See README.md and SOURCES.json for provenance, dependencies and known defects.

In [ ]:
import pandas as pd

file1 = open(r"./local-data/附件1.csv", encoding='utf-8')
file2 = open(r"./local-data/附件2.csv", encoding='utf-8')
file3 = open(r"./local-data/附件3.csv", encoding='utf-8')
file4 = open(r"./local-data/附件4.csv", encoding='utf-8')

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df4 = pd.read_csv(file4)

# 将 '销售日期' 列转换为日期时间格式
df2['销售日期'] = df2['销售日期'] + ' ' + df2['扫码销售时间']
df2['销售日期'] = pd.to_datetime(df2['销售日期'], format='%Y-%m-%d %H:%M:%S.%f')
df3['日期'] = pd.to_datetime(df3['日期'], format='%Y-%m-%d')
df3['日期'] = pd.to_datetime(df3['日期'], format='%Y-%m-%d')
df2 = df2.drop('扫码销售时间', axis=1)

# 选择2023/6/24之后的行数据
target_date = pd.to_datetime('2023-06-24')
filtered_df2 = df2[df2['销售日期'] >= target_date]
filtered_df3 = df3[df3['日期'] >= target_date]

df2 = filtered_df2.reset_index(drop=True)
df3 = filtered_df3.reset_index(drop=True)

print(filtered_df2.info())
print(df2.head())
print(df2.duplicated())
print('表三：', df3.info())




In [ ]:
## 确定24-30日的可售单品

code_of_veg = df2['单品编码'].unique()
num_of_veg = df2['单品编码'].nunique()
#print(df1['单品编码'].isin(code_of_veg))
selected_products = df1[df1['单品编码'].isin(code_of_veg)]
selected_product_names = selected_products['单品名称']
print('2023蔬菜单品数量为：', num_of_veg)
print('可售列表：', code_of_veg)
print('可售单品名称:', selected_product_names)
print('可售单品总数:', len(selected_product_names))



In [ ]:
on_sale = df2['是否打折销售'].unique()
#print(on_sale)
def trans(x):
    if x == df2['是否打折销售'].unique()[0]:
        return 0  # not on sale
    if x == df2['是否打折销售'].unique()[1]:
        return 1  # on sale
    
df2['是否打折销售'] = df2['是否打折销售'].apply(trans)
sale_type = df2['销售类型'].nunique()
print('销售类型有', sale_type, '种')

df2 = df2.drop('销售类型', axis=1)

print(df2.head())




In [ ]:
import matplotlib.pyplot as plt

#df2 = pd.concat([df2, pd.DataFrame(columns=['批发价格(元/千克)'])])

plt.rcParams['font.family'] = 'SimSun'

series = df2.set_index(['销售日期'], drop=True)
#series['销量(千克)'].plot()
#plt.show()

name_list = selected_product_names.tolist()
#date_list = [6.24, 6.25, 6.26, 6.27, 6.28, 6.29, 6.30]
code = []

for i in range(0, 49):
    name = name_list[i]
    code.append(code_of_veg[i])
    #批发价：whole_sale
    whole_sale = df3['批发价格(元/千克)'][df3['单品编码'].isin(code)]
    dates = df3['日期'][df3['单品编码'].isin(code)]
    plt.figure(figsize=(10, 6))
    plt.plot(dates, whole_sale, marker='o', linestyle='-')
    plt.title(name)
    plt.xlabel('日期')
    plt.ylabel('批发价格(元/千克)')
    plt.show()
    code = []
    #whole_sale.append



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras import layers
import datetime

#df2['销售日期'] = df2['销售日期'].apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d %H:%M:%S').timestamp())
#df2['销售日期'] = df2['销售日期'].astype(float).apply(lambda x: datetime.datetime.strptime(x, '%Y-%m-%d %H:%M:%S').timestamp())

#df2['销售日期'] = df2['销售日期'].dt.strftime('%Y-%m-%d %H:%M:%S')

df2['Unix时间戳'] = df2['销售日期'].apply(lambda x: x.timestamp())

# 划分训练集和测试集
#print(df2.head())
train_size = int(0.8 * len(df2))
train_data = df2[:train_size]
test_data = df2[train_size:]

#print(train_data.head())

# 创建 LSTM 模型
model = Sequential()
model.add(layers.LSTM(7, activation='relu', input_shape=(1, 4)))  # 输入形状为(7, 4)，表示过去7天的销售数据和特征
model.add(layers.Dense(1))
model.compile(optimizer='adam', loss='mse')

# 准备训练数据
X_train = train_data[['Unix时间戳', '单品编码', '销售单价(元/千克)', '是否打折销售']].values  # 特征列
y_train = train_data[['销量(千克)']].values  # 目标列
#print(len(X_train[0]))

X_train = X_train.reshape(2801, 1, 4)


#print(X_train)

scaler = MinMaxScaler()

# 归一化
#X_train_normalized = scaler.fit_transform(X_train)
#X_test_normalized = scaler.transform(X_test)

# 将数据转换为适合LSTM的形状
#X_train = X_train.reshape(-1, 7, 4)  # 7表示过去7天的数据，4表示特征数量

# 训练模型
model.fit(X_train, y_train, epochs=50, batch_size=32)

# 准备测试数据
X_test = test_data[['Unix时间戳', '单品编码', '销售单价(元/千克)', '是否打折销售']].values
X_test = X_test.reshape(-1, 1, 4)  # 同样需要将测试数据的形状转换为(7, 4)

y_pred = model.predict(X_test)
print(y_pred)
print(len(y_pred))




In [ ]:

# 数据归一化
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(product_data.values)

# 划分训练集和测试集
train_size = int(len(dataset) * 0.67)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size, :], dataset[train_size:len(dataset), :]

look_back = 7  # 基于过去7天的销售数据进行预测
trainX, trainY = create_dataset(train, look_back)
testX, testY = create_dataset(test, look_back)

# 将数据重塑为LSTM可接受的格式 [样本数, 时间步, 特征数]
trainX = np.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
testX = np.reshape(testX, (testX.shape[0], testX.shape[1], 1))

# 创建并训练LSTM模型
model = Sequential()
model.add(LSTM(50, input_shape=(look_back, 1)))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')
model.fit(trainX, trainY, epochs=100, batch_size=1, verbose=2)

# 预测销售数量
trainPredict = model.predict(trainX)
testPredict = model.predict(testX)

# 反归一化
trainPredict = scaler.inverse_transform(trainPredict)
trainY = scaler.inverse_transform([trainY])
testPredict = scaler.inverse_transform(testPredict)
testY = scaler.inverse_transform([testY])

# 计算均方根误差
trainScore = np.sqrt(mean_squared_error(trainY[0], trainPredict[:, 0]))
print('Train Score: %.2f RMSE' % trainScore)
testScore = np.sqrt(mean_squared_error(testY[0], testPredict[:, 0]))
print('Test Score: %.2f RMSE' % testScore)

# 使用模型进行未来一周的销售预测
last_week_sales = dataset[-look_back:]
next_week_dates = pd.date_range(start=product_data.index[-1], periods=7)
next_week_sales = []

for i in range(7):
    input_data = last_week_sales.reshape((1, look_back, 1))
    next_day_sales = model.predict(input_data)[0][0]
    next_week_sales.append(next_day_sales)
    last_week_sales = np.append(last_week_sales[1:], [[next_day_sales]], axis=0)

# 反归一化
next_week_sales = scaler.inverse_transform(np.array(next_week_sales).reshape(-1, 1))

# 输出未来一周的销售预测
next_week_sales_df = pd.DataFrame(next_week_sales, index=next_week_dates, columns=['销售数量'])
print(next_week_sales_df)

In [ ]:
print(type(df2['销售日期'].dtype))

In [ ]:
# 对于每个单品类别建立一个df,将这个df传入LSTM模型
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras import layers
import datetime

code = []
df5 = df2
for i in range(0, 49):
    code.append(code_of_veg[i])
    df5 = df5[df5['单品编码'].isin(code)]

print(df5.info())

df5['Unix时间戳'] = df5['销售日期'].apply(lambda x: x.timestamp())

# 划分训练集和测试集
train_size = int(0.8 * len(df5))
train_data = df5[:train_size]
test_data = df5[train_size:]

scaler = MinMaxScaler()

# 归一化

X_train_normalized =  scaler.fit_transform(train_data.iloc[:, 1:])
X_test_normalized = scaler.transform(test_data.iloc[:, 1:])

# 创建 LSTM 模型
model = Sequential()
model.add(LSTM(32, activation='relu', kernel_regularizer=l2(0.01), input_shape=(7, 4)))  # LSTM层，输入形状为(7, 特征数量)
model.add(Dense(1, activation='linear'))  # 输出层
model.compile(optimizer='adam', loss='mse')


# 准备训练数据
X_train = train_data[['Unix时间戳', '单品编码', '销售单价(元/千克)', '是否打折销售']].values  # 特征列
y_train = train_data[['销量(千克)']].values  # 目标列
print(len(X_train[0]))
print((X_train.shape))
print((y_train.shape))

X_train = X_train.reshape(-1, 7, 4)
y_train = y_train.reshape(-1, 7, 1)
print((X_train.shape))
print((y_train.shape))

model.fit(X_train, y_train, epochs=50, batch_size=32)

X_test = test_data[['Unix时间戳', '单品编码', '销售单价(元/千克)', '是否打折销售']].values  # 特征列
y_test = test_data[['销量(千克)']].values
print(X_test[0])
X_test = X_test.reshape(4, 7, 4)
#y_test = y_test.reshape(-1, 7, 1)

y_pred = model.predict(X_test)
print(y_pred)
print(len(y_pred))

    


In [ ]:
df5['Unix时间戳'] = df5['销售日期'].apply(lambda x: x.timestamp())

# 划分训练集和测试集
train_size = int(0.8 * len(df5))
train_data = df5[:train_size]
test_data = df5[train_size:]

# 创建 LSTM 模型
model = Sequential()
model.add(layers.LSTM(7, activation='relu', input_shape=(1, 4)))  # 输入形状为(7, 4)，表示过去7天的销售数据和特征
model.add(layers.Dense(1))
model.compile(optimizer='adam', loss='mse')

# 准备训练数据
X_train = train_data[['Unix时间戳', '单品编码', '销售单价(元/千克)', '是否打折销售']].values  # 特征列
y_train = train_data[['销量(千克)']].values  # 目标列
#print(len(X_train[0]))